# 🏭 AutoFactoryScope: YOLO11 Training (Google Drive)

**Dataset Location:** Already unzipped in Google Drive

**Path:** `/content/drive/MyDrive/AutoFactoryScope/robot_detection_layouts.v2i.yolov8 (1)`

**Runtime:** ~45-60 minutes on Colab T4 GPU

## ⚙️ Setup: GPU Check

In [ ]:
# Verify GPU availability
!nvidia-smi

## 📦 Install Dependencies

In [ ]:
# Install Ultralytics YOLO11
!pip install ultralytics -q

from ultralytics import YOLO
import torch
import yaml
from pathlib import Path
import shutil

print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

## 💾 Mount Google Drive & Load Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Drive mounted successfully!")

In [ ]:
# Copy dataset from Drive to Colab local storage (faster I/O)
source_path = '/content/drive/MyDrive/AutoFactoryScope/robot_detection_layouts.v2i.yolov8 (1)'
dest_path = '/content/dataset'

print(f"Copying dataset from Drive to Colab...")
print(f"  Source: {source_path}")
print(f"  Dest: {dest_path}")

shutil.copytree(source_path, dest_path)

print("\n✓ Dataset copied successfully!")

# Verify structure
!ls -la /content/dataset

In [ ]:
# Inspect data.yaml
!cat /content/dataset/data.yaml

## 🔧 Fix data.yaml Paths

In [ ]:
# Read and update data.yaml with absolute paths
data_yaml_path = Path('/content/dataset/data.yaml')

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Update paths to absolute
base_path = Path('/content/dataset')
data_config['train'] = str(base_path / 'train' / 'images')
data_config['val'] = str(base_path / 'valid' / 'images')
data_config['test'] = str(base_path / 'test' / 'images')

# Save updated config
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print("✓ Updated data.yaml:")
print(yaml.dump(data_config, default_flow_style=False))

## 🚀 Train YOLO11 Model

In [ ]:
# Initialize YOLO11 nano model
model = YOLO('yolo11n.pt')  # Downloads pretrained weights automatically

# Train
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=512,  # Match your backend tile size
    batch=16,   # Adjust if GPU runs out of memory (try 8)
    
    # Data Augmentation (critical for small datasets)
    hsv_h=0.015,  # Hue variation
    hsv_s=0.7,    # Saturation variation
    hsv_v=0.4,    # Value variation
    degrees=10.0, # Rotation ±10 degrees
    translate=0.1,
    scale=0.5,
    shear=0.0,
    flipud=0.5,   # Vertical flip (layouts can be inverted)
    fliplr=0.5,   # Horizontal flip
    mosaic=1.0,   # Mosaic augmentation
    mixup=0.0,
    
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    
    # Training Settings
    patience=10,  # Early stopping after 10 epochs without improvement
    save=True,
    save_period=10,  # Save checkpoint every 10 epochs
    cache=True,   # Cache images in RAM for faster training
    device=0,     # GPU 0
    workers=4,
    project='runs/train',
    name='robot_detector_v1',
    exist_ok=True,
    pretrained=True,
    verbose=True
)

print("\n✓ Training complete!")

## 📊 Evaluate Model

In [ ]:
# Load best model
best_model = YOLO('runs/train/robot_detector_v1/weights/best.pt')

# Validate on test set
metrics = best_model.val(data='/content/dataset/data.yaml')

print(f"\n=== Model Performance ===")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.p.mean():.3f}")
print(f"Recall:    {metrics.box.r.mean():.3f}")

# Target: mAP50 > 0.80 for good performance

## 🖼️ Visualize Predictions

In [ ]:
from IPython.display import Image, display
import glob

# Run inference on test images
test_results = best_model.predict(
    source='/content/dataset/test/images',
    conf=0.25,  # Confidence threshold
    save=True,
    project='runs/predict',
    name='test_predictions'
)

# Display first 5 predictions
pred_images = sorted(glob.glob('runs/predict/test_predictions/*.jpg'))[:5]

for img_path in pred_images:
    print(f"\n{Path(img_path).name}")
    display(Image(filename=img_path, width=800))

## 💾 Save Model to Google Drive

In [ ]:
# Create models folder in Drive
!mkdir -p "/content/drive/MyDrive/AutoFactoryScope/models"

# Copy best model to Drive
!cp runs/train/robot_detector_v1/weights/best.pt "/content/drive/MyDrive/AutoFactoryScope/models/"

# Also save last checkpoint
!cp runs/train/robot_detector_v1/weights/last.pt "/content/drive/MyDrive/AutoFactoryScope/models/"

print("✓ Models saved to Google Drive:")
print("  - MyDrive/AutoFactoryScope/models/best.pt")
print("  - MyDrive/AutoFactoryScope/models/last.pt")
print("\nNext: Download to your local machine for ONNX export.")

## 📥 Alternative: Direct Download from Colab

In [ ]:
# Uncomment to download directly from Colab
# from google.colab import files
# files.download('runs/train/robot_detector_v1/weights/best.pt')

## ✅ Training Complete!

**Next Steps:**

1. Download `best.pt` from Drive to:
   ```
   c:\Users\georgem\source\repos\AutoFactoryScope\models\best.pt
   ```

2. Run ONNX export notebook:
   ```
   notebooks/02_export_onnx.ipynb
   ```

3. Test in your backend:
   ```powershell
   uvicorn autofactoryscope_api.main:app --reload
   ```